# Session 1 — Why Your Threshold Is a Guess: Null Models

**Goal of this session:** stop asking "what does this network property look like" and start asking "is this network property actually there, or is it what any network with the same basic wiring budget would look like anyway".

*Network Neuroscience in Python, session 1 of 10. Continues from [Python for Neuroscience](https://github.com/saeedrafsharx/python-for-neuroscience), session 10.*

## Why this matters

In the first course, session 10 ended with a correlation matrix, a threshold you picked by eye, and a graph. That graph had a clustering coefficient, a few hub-looking nodes, maybe some visible modules. All of that is a description. None of it is a claim.

The question a reviewer will actually ask is: compared to what? A brain network's clustering coefficient of 0.4 means nothing on its own — you need to know what clustering coefficient a network with the *same number of nodes, edges, and degree sequence* would have if it were wired up without any real organisation. That comparison network is called a **null model**, and this session builds the machinery to make one properly.

## The toy network

We need something with a *known* answer before we trust a method on real data. So we build `generate_toy_network()`: a small network of 8–12 "regions" split into modules, with a tunable amount of within-module and between-module wiring, plus one optional node that bridges the modules — a stand-in for a connector hub.

This function is the backbone of sessions 1 through 6. It gets pasted again at the top of each of those notebooks (same pattern as `generate_toy_signal()` in the first course), so every session can stand alone.

**It is not a brain.** It is a graph with ground truth we invented ourselves, which is exactly what you want when you are checking whether a method works before trusting it on data where you don't know the answer.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


def generate_toy_network(n_per_module=5, n_modules=2, p_within=0.7, p_between=0.05,
                          add_connector=True, connector_frac=0.6, seed=0):
    """A small synthetic multi-region network with known modular structure.

    Nodes are split into `n_modules` equal-sized communities. Within a
    community, every pair of nodes is connected independently with
    probability `p_within`. Between two different communities, every pair
    is connected independently with probability `p_between`. If
    `add_connector` is True, one extra node is added and wired to a fixed
    fraction (`connector_frac`) of the members of every community — a
    stand-in for a hub that bridges modules rather than living inside one.

    Returns a networkx.Graph. Each node carries a 'module' attribute:
    0, 1, ... for the true communities, or the string 'connector'.
    """
    rng = np.random.default_rng(seed)
    G = nx.Graph()

    node_id = 0
    modules = []
    for m in range(n_modules):
        members = []
        for _ in range(n_per_module):
            G.add_node(node_id, module=m)
            members.append(node_id)
            node_id += 1
        modules.append(members)

    for m in range(n_modules):
        members = modules[m]
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if rng.random() < p_within:
                    G.add_edge(members[i], members[j])

    for m1 in range(n_modules):
        for m2 in range(m1 + 1, n_modules):
            for u in modules[m1]:
                for v in modules[m2]:
                    if rng.random() < p_between:
                        G.add_edge(u, v)

    if add_connector:
        connector = node_id
        G.add_node(connector, module="connector")
        n_link = max(1, round(connector_frac * n_per_module))
        for members in modules:
            chosen = rng.choice(members, size=min(n_link, len(members)), replace=False)
            for other in chosen:
                G.add_edge(connector, int(other))

    return G


G = generate_toy_network(seed=0)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("density:", round(nx.density(G), 3))
print("connected:", nx.is_connected(G))

## Drawing it

Same visual language we'll use for every network in this series: nodes coloured by module, the connector in its own colour, edge width and node size kept simple so this reads on a recorded screen. Large fonts, minimal clutter, one consistent layout style across all ten sessions.

In [ ]:
def draw_toy_network(G, ax=None, title=""):
    colours = {0: "#2b6cb0", 1: "#dd6b20", 2: "#805ad5", "connector": "#38a169"}
    node_colours = [colours[G.nodes[n]["module"]] for n in G.nodes()]
    node_sizes = [500 + 220 * G.degree(n) for n in G.nodes()]

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))
    pos = nx.spring_layout(G, seed=1)
    nx.draw_networkx_edges(G, pos, alpha=0.4, width=1.6, ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=node_colours, node_size=node_sizes,
                            edgecolors="white", linewidths=1.2, alpha=0.95, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=11, font_color="white",
                             font_weight="bold", ax=ax)
    ax.set_title(title, fontsize=13)
    ax.axis("off")
    return pos


fig, ax = plt.subplots(figsize=(7, 6))
draw_toy_network(G, ax=ax, title="The toy network (two modules + one connector)")
plt.tight_layout()
plt.show()

## One metric, on the real network

We'll use the **clustering coefficient**: for each node, the fraction of its neighbours that are also connected to each other, averaged over all nodes. High clustering means the network is full of tightly-knit triangles — friend-of-a-friend-is-a-friend structure. Brain networks are usually much more clustered than you'd expect by chance, which is one of the classic "small-world" findings in the field.

In [ ]:
observed_clustering = nx.average_clustering(G)
print(f"observed clustering coefficient: {observed_clustering:.3f}")

## What would "by chance" even mean?

The naive null model is an **Erdős–Rényi random graph**: same number of nodes and edges, but each edge placed completely at random. The problem is that this destroys the one thing that makes brain-like networks brain-like — a handful of high-degree hub nodes and a lot of low-degree ones. An Erdős–Rényi graph has a narrow, roughly-Poisson degree distribution. If you compare your network to that, and it looks different, you might just be rediscovering "my network has hubs and yours doesn't", which was never in question.

The fix is a **configuration model** (also called degree-preserving randomisation): shuffle *which* nodes are connected to which, while keeping every single node's degree exactly the same as in the real network. Now the only thing that can differ between the real network and the null is the *arrangement* of edges, not the *budget* of edges each node gets to spend. Any difference you find after that is about organisation, not about degree.

The simplest way to build one is repeated **double-edge swaps**: pick two edges (a–b) and (c–d), and if neither a–c nor b–d already exists, swap them to (a–c) and (b–d). Every node keeps its exact degree; the wiring gets scrambled. `networkx` does this as `nx.double_edge_swap`.

In [ ]:
def degree_preserving_null(G, n_swaps_per_edge=10, seed=0):
    """One randomised network with the same degree sequence as G."""
    rng = np.random.default_rng(seed)
    G_null = G.copy()
    n_swaps = max(10, n_swaps_per_edge * G.number_of_edges())
    nx.double_edge_swap(G_null, nswap=n_swaps, max_tries=n_swaps * 20,
                         seed=int(rng.integers(1_000_000_000)))
    return G_null


G_null_example = degree_preserving_null(G, seed=1)
same_degrees = sorted(dict(G.degree()).values()) == sorted(dict(G_null_example.degree()).values())
print("degree sequence preserved:", same_degrees)
print("edges preserved:", G.number_of_edges() == G_null_example.number_of_edges())

## Side by side

The real network on the left, one randomised version on the right. Same nodes, same degree for every node, same total number of edges — but the modular structure is gone. This is what "the same wiring budget, no organisation" looks like.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
draw_toy_network(G, ax=axes[0], title="Real toy network")
draw_toy_network(G_null_example, ax=axes[1], title="One degree-preserving randomisation")
plt.tight_layout()
plt.show()

## Building the null distribution

One randomised network isn't enough — randomness in the *randomisation* itself means the metric will bounce around. So we build an **ensemble**: many independent degree-preserving randomisations, and compute the metric on each. That gives us a distribution of "what this metric looks like when only degree is preserved", which we compare the real, observed value against.

In [ ]:
def null_distribution(G, metric_fn, n_random=500, n_swaps_per_edge=10, seed=0):
    """Compute metric_fn on n_random degree-preserving randomisations of G."""
    rng = np.random.default_rng(seed)
    values = np.empty(n_random)
    for i in range(n_random):
        G_null = degree_preserving_null(G, n_swaps_per_edge=n_swaps_per_edge,
                                         seed=int(rng.integers(1_000_000_000)))
        values[i] = metric_fn(G_null)
    return values


null_clustering = null_distribution(G, nx.average_clustering, n_random=500, seed=2)
print(f"null mean: {null_clustering.mean():.3f}   null std: {null_clustering.std():.3f}")
print(f"observed:  {observed_clustering:.3f}")

## The comparison

Now we can see whether the real network's clustering coefficient sits inside the cloud of "degree alone would produce this" values, or clearly outside it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.hist(null_clustering, bins=30, color="#a0aec0", edgecolor="white",
        label="null distribution\n(degree-preserving randomisations)")
ax.axvline(observed_clustering, color="#c53030", linewidth=3,
           label=f"observed = {observed_clustering:.3f}")
ax.set_xlabel("clustering coefficient", fontsize=12)
ax.set_ylabel("count (out of 500 randomisations)", fontsize=12)
ax.set_title("Is the real network's clustering more than degree alone explains?", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

z = (observed_clustering - null_clustering.mean()) / null_clustering.std()
p_perm = (np.sum(null_clustering >= observed_clustering) + 1) / (len(null_clustering) + 1)
print(f"z-score: {z:.2f}")
print(f"one-sided permutation p-value: {p_perm:.4f}")

## What just happened

The observed clustering coefficient sits well above the null cloud. That means the real toy network's clustering is not simply a side effect of a few nodes having high degree — the modular wiring we built into it is doing real work. On a real brain network, this is the difference between "this network has hubs" (true of almost any biological network and therefore not very interesting) and "this network is organised beyond what its hubs alone would produce" (the actual scientific claim).

**Next session:** we turn this one-off z-score into a reusable function, run it on several metrics at once, and deal honestly with what happens to your false-positive rate when you do that.